# Phase 0: Setup & Verification
Run this at the start of every Colab session.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os
import subprocess
REPO_DIR = '/content/bea2026-vocab-difficulty'
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
    subprocess.run(['git', 'pull'], check=False)
else:
    subprocess.run(['git', 'clone', 'https://github.com/YOUR_ORG/bea2026-vocab-difficulty.git', REPO_DIR], check=True)
    os.chdir(REPO_DIR)
print(os.getcwd())

In [ ]:
!pip install -r requirements.txt -q

In [ ]:
import shutil
BEA_DRIVE = '/content/drive/MyDrive/bea2026'
if os.path.isdir(BEA_DRIVE):
    for l1 in ['es', 'de', 'cn']:
        src_t = os.path.join(BEA_DRIVE, 'data', 'train', l1)
        dst_t = os.path.join(REPO_DIR, 'data', 'train', l1)
        if os.path.isdir(src_t):
            for f in os.listdir(src_t):
                if f.endswith('.csv'):
                    shutil.copy2(os.path.join(src_t, f), os.path.join(dst_t, f))
        src_d = os.path.join(BEA_DRIVE, 'data', 'dev', l1)
        dst_d = os.path.join(REPO_DIR, 'data', 'dev', l1)
        if os.path.isdir(src_d):
            for f in os.listdir(src_d):
                if f.endswith('.csv'):
                    shutil.copy2(os.path.join(src_d, f), os.path.join(dst_d, f))
    upstream_src = os.path.join(BEA_DRIVE, 'bea2026st')
    upstream_dst = os.path.join(REPO_DIR, 'upstream')
    if os.path.isdir(upstream_src):
        for f in ['evaluate.py', 'utils.py', 'finetune.py', 'predict.py', 'run_pipeline.py', 'download.py', 'environment.yml', 'model_parameters.csv']:
            p = os.path.join(upstream_src, f)
            if os.path.isfile(p):
                shutil.copy2(p, os.path.join(upstream_dst, f))
        baseline_dir = os.path.join(upstream_src, 'baseline_predictions_closed')
        if os.path.isdir(baseline_dir):
            os.makedirs(os.path.join(upstream_dst, 'baseline_predictions_closed'), exist_ok=True)
            for f in os.listdir(baseline_dir):
                if f.endswith('.csv'):
                    shutil.copy2(os.path.join(baseline_dir, f), os.path.join(upstream_dst, 'baseline_predictions_closed', f))
    print('Data and upstream copied from Drive.')
else:
    print('BEA_DRIVE not found; copy data and upstream manually.')

In [ ]:
assert not os.path.isdir(os.path.join(REPO_DIR, 'upstream', 'baseline_predictions_open')), 'No open track files.'
print('No open track baseline dir found (closed track only).')

In [ ]:
import torch
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('Device:', torch.cuda.get_device_name(0))

In [ ]:
import pandas as pd
TRAIN = {l1: f'data/train/{l1}/kvl_shared_task_{l1}_train.csv' for l1 in ['es','de','cn']}
DEV = {l1: f'data/dev/{l1}/kvl_shared_task_{l1}_dev.csv' for l1 in ['es','de','cn']}
for l1 in ['es','de','cn']:
    tr = pd.read_csv(TRAIN[l1])
    dv = pd.read_csv(DEV[l1])
    print(f'{l1} train: {tr.shape}, dev: {dv.shape}')

In [ ]:
import sys
sys.path.insert(0, os.path.join(REPO_DIR))
from scripts.evaluate import evaluate_from_files, CLOSED_BASELINES
base_dir = os.path.join(REPO_DIR, 'upstream', 'baseline_predictions_closed')
for l1 in ['es','de','cn']:
    pred = os.path.join(base_dir, f'kvl_shared_task_{l1}_dev_pred.csv')
    if not os.path.isfile(pred):
        pred = os.path.join(base_dir, f'{l1}_dev_pred.csv')
    gold = os.path.join(REPO_DIR, DEV[l1])
    if os.path.isfile(pred):
        evaluate_from_files(pred, gold, l1)
    else:
        print(f'L1: {l1} | Baseline pred file not found; expected RMSE: {CLOSED_BASELINES[l1]}')

## Summary
Setup complete: Drive mounted, repo ready, data and upstream copied, GPU verified, CSVs loaded. Official baseline RMSE: es=1.357, de=1.328, cn=1.175.